# ProjectMem Colab Runner

Edit the repo URL in the next cell, switch Colab to a GPU runtime, then run all cells.

Default benchmark set is fully auto-downloadable from Hugging Face: `mab_conflict`, `longmemeval`, `locomo`, `safeflow`.


In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"
BRANCH = "main"
WORKDIR = "/content/ProjectMem"

if os.path.isdir(WORKDIR):
    subprocess.run(["git", "-C", WORKDIR, "fetch", "origin"], check=False)
    subprocess.run(["git", "-C", WORKDIR, "checkout", BRANCH], check=False)
    subprocess.run(["git", "-C", WORKDIR, "pull", "origin", BRANCH], check=False)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
print("Working directory:", os.getcwd())
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip())


In [ ]:
%cd /content/ProjectMem
!python -m pip install -q --upgrade pip
!python -m pip install -q -r requirements.txt

import os
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face login complete.")
else:
    print("HF_TOKEN not set. Public benchmarks and public Qwen variants can still download.")


In [ ]:
import torch

BENCHMARKS = "mab_conflict,longmemeval,locomo,safeflow"
MAX_SCENARIOS = 1
USE_DUMMY = False
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DEVICE = "auto"
OUTPUT_ROOT = "reports/colab"

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import shlex
import subprocess
import sys

cmd = [
    sys.executable,
    "scripts/run_benchmark_matrix.py",
    "--benchmarks", BENCHMARKS,
    "--max-scenarios", str(MAX_SCENARIOS),
    "--output-root", OUTPUT_ROOT,
    "--device", DEVICE,
]

if USE_DUMMY:
    cmd.append("--use-dummy")
else:
    cmd.extend(["--agent1-model", MODEL_NAME, "--agent2-model", MODEL_NAME])

print("Running:")
print(shlex.join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
import json
import zipfile
from pathlib import Path

output_root = Path(OUTPUT_ROOT)
summary_path = output_root / "matrix_summary.json"

if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))

for path in sorted(output_root.rglob("*.json")):
    print(path)

zip_path = Path("/content/projectmem_reports.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as archive:
    for file_path in output_root.rglob("*"):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(output_root.parent))

print("Created:", zip_path)
